# IndicQA - Difficulty Assignment

Assigns `difficulty` (Easy / Medium / Hard) to the two IndicQA reading-
comprehension files.

| file | rows | scored | languages | task | answer |
|---|---|---|---|---|---|
| `indicqa_extractive.jsonl` | 1100 | 400 | 11 | `SHORT_ANSWER` | a span, median 2 words |
| `indicqa_abstractive.jsonl` | 900 | 400 | 9 | `GENERATIVE` | composed, median 11 words |

**400 rows per split, stratified by language** - roughly 36 per language for
extractive and 44 for abstractive. A pooled draw of 400 would leave one
language on 28 and another on 45 purely by chance, which makes the per-language
table in Cell 13 noise rather than signal. Set `STRATIFY = None` for a plain
draw, or `N_ROWS = None` to score everything.

**These are open-book tasks, and the passage lives in `explanation`.** The
14-key schema has no context field, so that is where it ended up. The evidence
is unambiguous: in the extractive file the answer's tokens are covered by
`explanation` at a **median of 100%**, and the answer appears verbatim inside
it on 68% of rows, against a median passage of 728 characters and a median
answer of 12.

So the passage is fed to the model as context. Scoring these closed-book
would turn extractive QA into open-domain trivia and produce a near-100% Hard
column - a dead label, the same outcome `comi_lingua` and `heritage` reached
for their own reasons.

**Judge.** `Qwen/Qwen3-14B`, independent of all three answerers - a different
family and larger than any of them - grading 0-4 against the reference. The
grade is read from the logits of the digit tokens in one forward pass, so
there is no free text to parse. Raw grades are stored; Cell 11 applies the
threshold, so re-thresholding never re-runs a model.

| Models passing | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

**Two floors are measured before any label is trusted** (Cell 4, no model
involved):

- **copy-the-passage** - submit the passage itself as the answer. This matters
  for the abstractive file, where **35 rows have an answer at least 90% the
  length of their own passage and 6 are byte-identical to it**. On those rows
  copying scores full marks, and a label derived from them measures nothing.
- **the wrong answer** - submit another row's answer. This is the floor a real
  attempt has to clear.

**Long passages are windowed, not dropped.** 49 extractive passages exceed
4000 characters and one is 91,670. `window_passage` keeps the region around
the best lexical match for the answer, so the span survives while the prompt
stays inside the context limit.

**Output.** `indicqa_<split>_difficulty.jsonl` with all 14 schema fields, plus
an audit file holding every model's answer and every raw grade.

**Runtime.** 400 rows x 3 models = 1200 generations, then 1200 gradings plus
400 floor gradings on one judge load - roughly 2-3 hours per split on a free
T4. Progress is written every batch, so a disconnect resumes where it
stopped.

### Cell 1 - Install dependencies and authenticate

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; Mistral and
Qwen3 are open. Accept each licence on huggingface.co, create a **read**
token, then add it in Colab via the **key icon** as a secret named `HF_TOKEN`.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("HF login skipped ({}). Gated models will fail to load.".format(e))

### Cell 2 - Mount Drive

Four models are loaded over a full run - three answerers and the judge - so
caching 4-bit copies to Drive saves roughly 30 GB of download on every rerun.

In [ ]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights will NOT be cached between sessions.")

### Cell 3 - Configuration

- `SPLIT` - `"extractive"` or `"abstractive"`. Every path is derived from it,
  so the two runs never collide and both can share one Drive folder.
- `N_ROWS` / `STRATIFY` - 400 rows drawn evenly across languages. `N_ROWS =
  None` scores the whole file; `STRATIFY = None` makes it one pooled draw.
- `RUBRIC` - the 0-4 grading scale. Grades are stored raw.
- `THRESHOLD_MODE` - `"fixed"` (default, `0.625` = grade 2.5 of 4, i.e. better
  than "partially correct"), `"floor_margin"` to anchor above the measured
  wrong-answer floor, or `"auto_median"`.
- `MAX_PASSAGE_CHARS` - the window `window_passage` keeps around the answer.

In [ ]:
import gc
import os
import json
import random
import shutil
import statistics
from collections import Counter, defaultdict

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- which split ----
SPLIT = "extractive"          # "extractive" | "abstractive"

SPLITS = {
    "extractive": {
        "file": "indicqa_extractive.jsonl",
        "task": ("Answer the question using only the passage. Reply with the "
                 "shortest span from the passage that answers it - usually a "
                 "name, a number or a few words. Do not explain."),
        "note": "answer is a span; median 2 words",
    },
    "abstractive": {
        "file": "indicqa_abstractive.jsonl",
        "task": ("Answer the question using only the passage. Reply with one "
                 "or two sentences in the same language as the passage. Do not "
                 "copy the passage wholesale and do not add outside "
                 "information."),
        "note": "answer is composed; median 11 words",
    },
}
assert SPLIT in SPLITS, "SPLIT must be one of {}".format(list(SPLITS))
CFG = SPLITS[SPLIT]

# ---- paths, all derived from SPLIT ----
INPUT_FILE  = CFG["file"]
OUTPUT_FILE = "indicqa_{}_difficulty.jsonl".format(SPLIT)
AUDIT_FILE  = "indicqa_{}_audit.jsonl".format(SPLIT)
GEN_DIR     = "gen_progress_{}".format(SPLIT)      # one file per answerer
JUDGE_FILE  = "judge_progress_{}.jsonl".format(SPLIT)

# ---- sampling ----
N_ROWS   = 400           # None = the whole file
STRATIFY = "language"    # None = one pooled draw
SEED     = 42

# ---- passage handling ----
MAX_PASSAGE_CHARS = 3000

# ---- grading ----
RUBRIC = {
    0: "wrong or irrelevant",
    1: "mostly wrong, only slight overlap",
    2: "partially correct, misses key points",
    3: "mostly correct, minor omissions",
    4: "fully correct",
}
MAX_GRADE = max(RUBRIC)

THRESHOLD_MODE  = "fixed"          # fixed | floor_margin | auto_median
FIXED_THRESHOLD = 0.625            # = grade 2.5 / 4
FLOOR_MARGIN    = 0.15

# ---- generation ----
MAX_NEW_TOKENS = 48 if SPLIT == "extractive" else 160
BATCH_SIZE     = 25

# ---- schema ----
SET_EVAL_METRIC = "llm_as_a_judge"

# ---- models ----
GENERATORS = [
    {"name": "mistral", "repo": "mistralai/Mistral-7B-Instruct-v0.3"},   # ungated
    {"name": "llama",   "repo": "meta-llama/Llama-3.1-8B-Instruct"},     # GATED
    {"name": "gemma",   "repo": "google/gemma-2-9b-it", "attn": "eager"},# GATED
]

# independent of all three: different family, larger, ungated
JUDGE = {"name": "qwen3judge", "repo": "Qwen/Qwen3-14B"}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(GEN_DIR, exist_ok=True)
print("split :", SPLIT, "-", CFG["note"])
print("input :", INPUT_FILE)
print("output:", OUTPUT_FILE)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

### Cell 4 - Load, window the passages, measure the floors

`window_passage` only acts on passages over `MAX_PASSAGE_CHARS`. It slides a
window and keeps the one with the most answer-token overlap, so the span the
question is about survives the trim. Rows whose passage no longer contains the
answer after windowing are reported, not silently kept.

Then two floors, neither of which loads a model:

- **copy-the-passage** - grade the passage itself as if it were the answer.
  High here means a lazy model wins, which is a property of the data.
- **wrong answer** - another row's answer against this row's reference. This
  is the level a genuine attempt must clear.

Both are reported per language, because a floor that binds in one script can
be irrelevant in another - the lesson from the CORIL and COMTAIL pairs.

In [ ]:
import re

with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f if line.strip()]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))


def toks(s):
    return re.findall(r"\w+", str(s))


def window_passage(passage, answer, limit=MAX_PASSAGE_CHARS):
    """Keep a `limit`-char window that still contains the answer.

    A verbatim answer is centred exactly, which a sliding grid cannot
    guarantee - a coarse step can straddle the span and cut it in half.
    Only when the answer does not appear literally does this fall back to
    scanning for the window with the most answer-token overlap."""
    passage, ans = str(passage), str(answer).strip()
    if len(passage) <= limit:
        return passage

    at = passage.find(ans)
    if ans and at >= 0 and len(ans) <= limit:
        start = max(0, min(at - (limit - len(ans)) // 2, len(passage) - limit))
        return passage[start:start + limit]

    want = set(toks(answer))
    if not want:
        return passage[:limit]
    step, best, best_hit = max(limit // 16, 1), passage[:limit], -1
    for start in range(0, max(len(passage) - limit, 0) + step, step):
        chunk = passage[start:start + limit]
        hit = len(want & set(toks(chunk)))
        if hit > best_hit:
            best, best_hit = chunk, hit
    return best


rows, windowed, lost = [], 0, []
for r in all_rows:
    p = str(r.get("explanation") or "")
    w = window_passage(p, r["answer"])
    if len(w) < len(p):
        windowed += 1
        if str(r["answer"]).strip() in p and str(r["answer"]).strip() not in w:
            lost.append(r["id"])
    r["_passage"] = w
    rows.append(r)

print("  passages windowed to {} chars: {}".format(MAX_PASSAGE_CHARS, windowed))
print("  windows that lost a verbatim answer: {} {}".format(len(lost), lost[:3]))

# rows where the reference IS the passage - copying scores full marks
degenerate = [r for r in rows
              if len(str(r["answer"])) >= 0.9 * max(len(str(r["_passage"])), 1)]
print("  rows whose answer is >=90% the length of its passage: {}".format(len(degenerate)))
print("    of those, byte-identical to the passage: {}".format(
    sum(1 for r in degenerate if str(r["answer"]).strip() == str(r["_passage"]).strip())))
if degenerate:
    print("    -> on these, copying the passage is a perfect answer. They are")
    print("       kept, flagged in the audit as `degenerate`, and reported")
    print("       separately in Cell 12. Set DROP_DEGENERATE to exclude them.")

DROP_DEGENERATE = False
if DROP_DEGENERATE:
    keep = {r["id"] for r in degenerate}
    rows = [r for r in rows if r["id"] not in keep]
    print("    dropped {} degenerate rows".format(len(keep)))

random.seed(SEED)
if N_ROWS is None:
    sample = rows
elif not STRATIFY:
    sample = random.sample(rows, min(N_ROWS, len(rows)))
else:
    # even quota per stratum, remainder handed out largest-group-first so the
    # total lands exactly on N_ROWS and every language is represented
    groups = defaultdict(list)
    for r in rows:
        groups[r[STRATIFY]].append(r)
    keys  = sorted(groups, key=lambda k: (-len(groups[k]), k))
    base, extra = divmod(min(N_ROWS, len(rows)), len(keys))
    sample = []
    for i, k in enumerate(keys):
        want = min(base + (1 if i < extra else 0), len(groups[k]))
        sample.extend(random.sample(groups[k], want))
    # if a stratum was too small to fill its quota, top up from what is left
    short = min(N_ROWS, len(rows)) - len(sample)
    if short > 0:
        taken = {r["id"] for r in sample}
        sample.extend(random.sample([r for r in rows if r["id"] not in taken], short))
    random.shuffle(sample)

print("\nScoring {} of {} rows{}".format(
    len(sample), len(rows), " (stratified by " + STRATIFY + ")" if STRATIFY and N_ROWS else ""))
print("  languages: {}".format(dict(sorted(Counter(r["language"] for r in sample).items()))))
print("  sources  : {}".format(dict(Counter(r["source"] for r in sample).most_common())))

# ---- lexical floors (no model) ----
def overlap_score(hyp, ref):
    """Token-F1 - a cheap stand-in for the judge, used only for the floors."""
    h, g = Counter(toks(hyp)), Counter(toks(ref))
    inter = sum((h & g).values())
    if not inter:
        return 0.0
    p, r = inter / max(sum(h.values()), 1), inter / max(sum(g.values()), 1)
    return 2 * p * r / (p + r)


shuffled = sample[:]
random.shuffle(shuffled)
copy_floor, wrong_floor = defaultdict(list), defaultdict(list)
for i, r in enumerate(sample):
    copy_floor[r["language"]].append(overlap_score(r["_passage"], r["answer"]))
    other = shuffled[i]
    if other["id"] != r["id"]:
        wrong_floor[r["language"]].append(overlap_score(other["answer"], r["answer"]))

print("\nlexical floors (token-F1, no model involved):")
print("  {:<6} {:>6} {:>12} {:>12}".format("lang", "n", "copy-passage", "wrong-answer"))
for lang in sorted(copy_floor):
    print("  {:<6} {:>6} {:>11.1f}% {:>11.1f}%".format(
        lang, len(copy_floor[lang]),
        100 * statistics.mean(copy_floor[lang]),
        100 * statistics.mean(wrong_floor[lang]) if wrong_floor[lang] else 0.0))
ALL_COPY  = statistics.mean([v for l in copy_floor for v in copy_floor[l]])
ALL_WRONG = statistics.mean([v for l in wrong_floor for v in wrong_floor[l]])
print("  {:<6} {:>6} {:>11.1f}% {:>11.1f}%".format(
    "ALL", len(sample), 100 * ALL_COPY, 100 * ALL_WRONG))
print("\n  These are lexical, not the judge's scale. Read them as a warning")
print("  about the data: a high copy-passage figure means a model that")
print("  regurgitates the context is rewarded.")

r = sample[0]
print("\n--- example row ---")
print("  [{} / {}] {}".format(r["language"], r["source"], r["question"][:88]))
print("  answer  : {}".format(str(r["answer"])[:110]))
print("  passage : {}".format(r["_passage"][:160].replace("\n", " ")))

### Cell 5 - Build the prompt

The passage comes first and the question last, so the model reads the context
before it knows what is being asked - which is the ordering reading-
comprehension models are trained on. The task line differs between the two
splits: extractive asks for the shortest span, abstractive for one or two
sentences in the passage's language.

In [ ]:
def build_query(row):
    return ("{}\n\nPassage:\n{}\n\nQuestion:\n{}\n\nAnswer:").format(
        CFG["task"], row["_passage"], " ".join(str(row["question"]).split()))


SYSTEM_PROMPT = (
    "You answer reading-comprehension questions about Indian-language "
    "passages.\n\n"
    "Use only the passage. Answer in the same language and script as the "
    "passage. If the passage does not contain the answer, say so in that "
    "language rather than guessing."
)


def build_completion(row):
    return SYSTEM_PROMPT + "\n\n" + build_query(row)


def build_chat_messages(row):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_query(row)}]


demo = build_completion(sample[0])
print("=" * 70)
print(demo[:1200] + ("\n... [{} chars total]".format(len(demo)) if len(demo) > 1200 else ""))
print("=" * 70)
print("[reference: {}]".format(str(sample[0]["answer"])[:100]))

### Cell 6 - Generation and judging

`generate` produces the answer. `grade` hands the question, the reference and
the candidate to Qwen3 and reads a single digit **from the logits** rather
than from generated text - Qwen3 is a hybrid-reasoning model and would
otherwise open a `<think>` block, making the next token a tag instead of a
grade.

The judge prompt states explicitly that a different language or different
wording is acceptable when the content matches, which matters because the
references are Indic and the answerers sometimes reply in English. An empty
answer is graded 0 without troubling the judge.

In [ ]:
def prompt_style(tokenizer):
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def format_prompt(tokenizer, row):
    if prompt_style(tokenizer) == "completion":
        return build_completion(row)
    msgs = build_chat_messages(row)
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        # some templates (Gemma) reject a system role - fold it into the user
        # turn rather than dropping the instructions
        merged = [{"role": "user",
                   "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}]
        return tokenizer.apply_chat_template(
            merged, tokenize=False, add_generation_prompt=True)


def flat(s):
    return " ".join(str(s).split())


@torch.no_grad()
def generate(model, tokenizer, row):
    text   = format_prompt(tokenizer, row)
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=3584).to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                         do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    new = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()


JUDGE_INSTRUCTIONS = (
    "You are grading answers to reading-comprehension questions about "
    "passages written in Indian languages.\n\n"
    "You are given the question, the official reference answer, and a student "
    "answer. Grade how well the student answer matches the reference in "
    "FACTUAL CONTENT and MEANING. Ignore differences in wording, length, "
    "phrasing and writing style. An answer in a different language or script "
    "from the reference is fine if the content is correct. A longer answer "
    "that contains the reference content is fine.\n\n"
    "Grades:\n"
    + "\n".join("{} = {}".format(k, v) for k, v in sorted(RUBRIC.items()))
    + "\n\nReply with a single digit and nothing else."
)


def judge_user_turn(question, reference, answer):
    return ("Question:\n{}\n\nReference answer:\n{}\n\nStudent answer:\n{}"
            "\n\nGrade (0-{}):").format(
                flat(question), flat(reference), flat(answer), MAX_GRADE)


def judge_prompt(tokenizer, question, reference, answer):
    msgs = [{"role": "system", "content": JUDGE_INSTRUCTIONS},
            {"role": "user",   "content": judge_user_turn(question, reference, answer)}]
    try:
        # without enable_thinking=False Qwen3 opens a <think> block and the
        # next token is a tag, not a digit
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
            enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        merged = [{"role": "user",
                   "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}]
        return tokenizer.apply_chat_template(
            merged, tokenize=False, add_generation_prompt=True)


def digit_token_ids(tokenizer):
    ids = {}
    for g in sorted(RUBRIC):
        variants = set()
        for form in (str(g), " " + str(g)):
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                variants.add(enc[0])
        ids[g] = sorted(variants)
    return ids


@torch.no_grad()
def grade(model, tokenizer, tok_ids, question, reference, answer):
    if not str(answer).strip():
        return 0, 0.0                      # nothing to grade

    text   = judge_prompt(tokenizer, question, reference, answer)
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=3584).to(model.device)
    logits = model(**inputs).logits[0, -1]

    per_digit = torch.tensor(
        [max(logits[i].item() for i in tok_ids[g]) for g in sorted(RUBRIC)])
    probs  = torch.softmax(per_digit, dim=0)
    grades = torch.tensor([float(g) for g in sorted(RUBRIC)])

    hard = int(grades[int(torch.argmax(per_digit))].item())
    soft = float((probs * grades).sum().item()) / MAX_GRADE   # 0..1
    return hard, soft


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()

print("Generation and judging functions defined")

### Cell 7 - Load-or-cache

A Drive copy is already 4-bit, so passing `quantization_config` again would
quantise twice - the branch below avoids it. `trust_remote_code=False`
throughout: all four are native architectures in `transformers`, and the flag
has caused loader failures elsewhere in this project.

In [ ]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer

print("Loader defined")

### Cell 8 - Answer with all three models

Model-outer, row-inner: three loads for the whole run. Every batch is appended
to `gen_progress_<split>/<model>.jsonl` and read back on start, so a Colab
disconnect costs at most one batch and a model that already finished is never
loaded.

In [ ]:
def run_generator(spec, rows):
    prog = os.path.join(GEN_DIR, spec["name"] + ".jsonl")

    done = {}
    if os.path.exists(prog):
        with open(prog, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item["answer"]
        print("  resuming - {}/{} already answered".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    model, tokenizer = load_model(spec)
    total = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for start in range(0, len(remaining), BATCH_SIZE):
        batch, results = remaining[start:start + BATCH_SIZE], []
        for row in batch:
            results.append({"id": row["id"], "answer": generate(model, tokenizer, row)})
        with open(prog, "a", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        done.update({i["id"]: i["answer"] for i in results})
        empty = sum(1 for v in done.values() if not v.strip())
        print("  batch {}/{} saved - {}/{} rows | empty {}".format(
            start // BATCH_SIZE + 1, total, len(done), len(rows), empty))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done


generations = {}
for spec in GENERATORS:
    print("\n=== {} ===".format(spec["name"]))
    generations[spec["name"]] = run_generator(spec, sample)

print("\nAll generators done")

### Cell 9 - Judge every answer

One load of Qwen3-14B grades all three models' answers, plus the two floor
baselines on a 200-row subsample - so the floors land on the judge's own scale
rather than the lexical proxy from Cell 4. That is the number the threshold
should be compared against.

In [ ]:
FLOOR_SAMPLE = 200

def judge_all(rows):
    done = {}
    if os.path.exists(JUDGE_FILE):
        with open(JUDGE_FILE, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["key"]] = item
        print("  resuming - {} gradings already done".format(len(done)))

    jobs = []
    for row in rows:
        for spec in GENERATORS:
            key = "{}||{}".format(row["id"], spec["name"])
            if key not in done:
                jobs.append((key, row, generations[spec["name"]].get(row["id"], "")))

    random.seed(SEED)
    floor_rows = random.sample(rows, min(FLOOR_SAMPLE, len(rows)))
    shuffled = floor_rows[:]
    random.shuffle(shuffled)
    for i, row in enumerate(floor_rows):
        k = "{}||__copy__".format(row["id"])
        if k not in done:
            jobs.append((k, row, row["_passage"]))
        k = "{}||__wrong__".format(row["id"])
        if k not in done and shuffled[i]["id"] != row["id"]:
            jobs.append((k, row, shuffled[i]["answer"]))

    if not jobs:
        print("  judging already complete - skipping load")
        return done

    model, tokenizer = load_model(JUDGE)
    tok_ids = digit_token_ids(tokenizer)
    total = (len(jobs) + BATCH_SIZE - 1) // BATCH_SIZE

    for start in range(0, len(jobs), BATCH_SIZE):
        batch, results = jobs[start:start + BATCH_SIZE], []
        for key, row, answer in batch:
            hard, soft = grade(model, tokenizer, tok_ids,
                               row["question"], row["answer"], answer)
            results.append({"key": key, "grade": hard, "soft": soft})
        with open(JUDGE_FILE, "a", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        done.update({i["key"]: i for i in results})
        print("  batch {}/{} saved - {}/{} gradings".format(
            start // BATCH_SIZE + 1, total, len(done), len(jobs) + len(done) - len(jobs)))

    del model, tokenizer
    clear_hf_cache()
    print("  judging complete")
    return done


print("=== judge ===")
grades = judge_all(sample)

### Cell 10 - The judge's own floors

The two baselines from Cell 9, now on the 0-1 scale the threshold uses. Read
them before Cell 11: a threshold below the wrong-answer floor lets a model
pass by producing something unrelated, and a copy-passage floor near the top
of the scale means the split rewards regurgitation.

In [ ]:
copy_soft  = [v["soft"] for k, v in grades.items() if k.endswith("||__copy__")]
wrong_soft = [v["soft"] for k, v in grades.items() if k.endswith("||__wrong__")]

def describe(name, vals):
    if not vals:
        print("  {:<24} (not measured)".format(name)); return None
    print("  {:<24} mean {:.3f}  median {:.3f}  p90 {:.3f}  n={}".format(
        name, statistics.mean(vals), statistics.median(vals),
        sorted(vals)[int(0.9 * len(vals)) - 1], len(vals)))
    return statistics.mean(vals)

print("judge-scale floors (0-1, where 1.0 = grade 4):")
COPY_FLOOR  = describe("copy the passage", copy_soft)  or 0.0
WRONG_FLOOR = describe("an unrelated answer", wrong_soft) or 0.0
BINDING_FLOOR = max(COPY_FLOOR, WRONG_FLOOR)
print("\n  BINDING FLOOR = {:.3f}".format(BINDING_FLOOR))
if COPY_FLOOR >= WRONG_FLOOR:
    print("  -> copying the passage is the stronger baseline. A model that")
    print("     echoes the context beats one that answers badly, so treat")
    print("     scores near this level as non-answers.")

### Cell 11 - Threshold and difficulty

The threshold turns a continuous grade into a pass, votes sum, and the label
follows. Because every raw grade is on disk, changing `THRESHOLD_MODE` and
re-running this cell alone re-labels the whole file in seconds - no model is
touched. The sensitivity table shows what every other choice would have given.

In [ ]:
per_model = {s["name"]: [] for s in GENERATORS}
for row in sample:
    for s in GENERATORS:
        g = grades.get("{}||{}".format(row["id"], s["name"]))
        if g:
            per_model[s["name"]].append(g["soft"])

pooled = sorted(v for vals in per_model.values() for v in vals)
def pct(q):
    return pooled[min(int(q * len(pooled)), len(pooled) - 1)] if pooled else 0.0

print("grade distribution per model (0-1):")
print("  {:<10} {:>7} {:>8} {:>7} {:>7}".format("model", "p25", "median", "p75", "mean"))
for name, vals in per_model.items():
    if vals:
        v = sorted(vals)
        print("  {:<10} {:>6.3f} {:>8.3f} {:>7.3f} {:>7.3f}".format(
            name, v[len(v) // 4], statistics.median(v), v[3 * len(v) // 4],
            statistics.mean(v)))

if THRESHOLD_MODE == "fixed":
    THRESHOLD, why = FIXED_THRESHOLD, "fixed = grade {:.1f}/4".format(
        FIXED_THRESHOLD * MAX_GRADE)
elif THRESHOLD_MODE == "floor_margin":
    THRESHOLD, why = BINDING_FLOOR + FLOOR_MARGIN, "binding floor + {:.2f}".format(FLOOR_MARGIN)
elif THRESHOLD_MODE == "auto_median":
    THRESHOLD, why = pct(.50), "median of all pooled grades"
else:
    raise ValueError("unknown THRESHOLD_MODE")

print("\nsensitivity - what each threshold would produce:")
print("  {:>10} {:>7} {:>8} {:>6}".format("threshold", "Easy", "Medium", "Hard"))
for t in sorted({0.25, 0.375, 0.5, 0.625, 0.75, round(BINDING_FLOOR + FLOOR_MARGIN, 3),
                 round(THRESHOLD, 3)}):
    c = Counter()
    for row in sample:
        v = sum(1 for s in GENERATORS
                if grades.get("{}||{}".format(row["id"], s["name"]), {}).get("soft", 0) >= t)
        c["Easy" if v == 3 else "Medium" if v == 2 else "Hard"] += 1
    mark = "  <- CHOSEN" if abs(t - THRESHOLD) < 1e-9 else ""
    mark += "  (binding floor)" if abs(t - BINDING_FLOOR) < 1e-9 else ""
    print("  {:>10.3f} {:>7} {:>8} {:>6}{}".format(t, c["Easy"], c["Medium"], c["Hard"], mark))

print("\nTHRESHOLD = {:.3f}  ({})".format(THRESHOLD, why))
if THRESHOLD < BINDING_FLOOR:
    print("  WARNING: below the binding floor ({:.3f}). A model could pass".format(BINDING_FLOOR))
    print("  without answering. Use THRESHOLD_MODE='floor_margin'.")

### Cell 12 - Assign difficulty and write the schema

The row is projected onto the 14 schema keys in order, so `_passage` and the
other working fields never reach the output - and `explanation` keeps the
**full original passage**, not the window, since the window is an artefact of
this run's context limit.

In [ ]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    if score == 2:
        return "Medium"
    return "Hard"


degenerate_ids = {r["id"] for r in degenerate}
final_results, audit = [], []

for row in sample:
    soft  = [grades.get("{}||{}".format(row["id"], s["name"]), {}).get("soft", 0.0)
             for s in GENERATORS]
    hard  = [grades.get("{}||{}".format(row["id"], s["name"]), {}).get("grade", 0)
             for s in GENERATORS]
    votes = [int(v >= THRESHOLD) for v in soft]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":         row["id"],
        "split":      SPLIT,
        "difficulty": difficulty,
        "votes":      votes,
        "soft":       [round(v, 4) for v in soft],
        "grade":      hard,
        "language":   row["language"],
        "source":     row["source"],
        "degenerate": row["id"] in degenerate_ids,
        "windowed":   len(row["_passage"]) < len(str(row.get("explanation") or "")),
        "question":   flat(row["question"])[:300],
        "reference":  flat(row["answer"])[:300],
        "answers":    {s["name"]: flat(generations[s["name"]].get(row["id"], ""))[:300]
                       for s in GENERATORS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))
print("threshold used: {:.3f}".format(THRESHOLD))

### Cell 13 - Verify and report

Nothing here rebalances anything; it only makes the result legible.

- **Schema**: 14 keys in order, no null difficulty, and question / answer /
  explanation still byte-identical to the input file.
- **Distribution** overall, per language and per source dataset.
- **The degenerate rows reported separately.** If their Easy rate is far above
  the rest, the split is rewarding models for copying and those labels should
  not be used.
- **Per-model means against the binding floor**, so a model that never cleared
  the floor is visible rather than silently counted as a vote.

In [ ]:
bad_keys  = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
null_diff = [r["id"] for r in final_results if not r["difficulty"]]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {}".format(
    len(final_results), len(bad_keys), len(null_diff)))

src = {r["id"]: r for r in all_rows}
altered = [r["id"] for r in final_results
           if json.dumps([r["question"], r["answer"], r["explanation"]], ensure_ascii=False)
           != json.dumps([src[r["id"]]["question"], src[r["id"]]["answer"],
                          src[r["id"]]["explanation"]], ensure_ascii=False)]
print("question/answer/explanation altered: {}".format(len(altered)))

dist = Counter(r["difficulty"] for r in final_results)
print("\nDifficulty distribution (threshold {:.3f}):".format(THRESHOLD))
for lvl in ("Easy", "Medium", "Hard"):
    print("  {:<7}: {:5}  ({:.1%})".format(lvl, dist[lvl], dist[lvl] / len(final_results)))

print("\nPer model (mean grade vs binding floor {:.3f}):".format(BINDING_FLOOR))
for s in GENERATORS:
    vals = per_model[s["name"]]
    if vals:
        m = statistics.mean(vals)
        flag = "  <- at/below the floor" if m <= BINDING_FLOOR else ""
        print("  {:<10} {:.3f} | empty {}/{}{}".format(
            s["name"], m,
            sum(1 for v in generations[s["name"]].values() if not v.strip()),
            len(sample), flag))

print("\nPer language:")
tab = defaultdict(Counter)
for a in audit:
    tab[a["language"]][a["difficulty"]] += 1
print("  {:<6} {:>5} {:>6} {:>7} {:>6}   %Hard".format("lang", "n", "Easy", "Medium", "Hard"))
for k in sorted(tab, key=lambda k: -tab[k]["Hard"] / max(sum(tab[k].values()), 1)):
    c = tab[k]; n = sum(c.values())
    print("  {:<6} {:>5} {:>6} {:>7} {:>6}   {:.0f}%".format(
        k, n, c["Easy"], c["Medium"], c["Hard"], 100 * c["Hard"] / n))

print("\nPer source dataset:")
tab2 = defaultdict(Counter)
for a in audit:
    tab2[a["source"]][a["difficulty"]] += 1
for k in sorted(tab2, key=lambda k: -sum(tab2[k].values())):
    c = tab2[k]; n = sum(c.values())
    print("  {:<14} {:>5}  E{:>4} M{:>4} H{:>4}   {:.0f}% Hard".format(
        k, n, c["Easy"], c["Medium"], c["Hard"], 100 * c["Hard"] / n))

deg = [a for a in audit if a["degenerate"]]
if deg:
    c = Counter(a["difficulty"] for a in deg)
    rest = Counter(a["difficulty"] for a in audit if not a["degenerate"])
    print("\nRows whose answer is >=90% of its passage ({} of {}):".format(
        len(deg), len(audit)))
    print("  those rows   : E{} M{} H{}  ({:.0f}% Easy)".format(
        c["Easy"], c["Medium"], c["Hard"], 100 * c["Easy"] / len(deg)))
    print("  everything else: E{} M{} H{}  ({:.0f}% Easy)".format(
        rest["Easy"], rest["Medium"], rest["Hard"],
        100 * rest["Easy"] / max(sum(rest.values()), 1)))
    print("  A much higher Easy rate on the first line means those labels")
    print("  record copying, not comprehension. Set DROP_DEGENERATE in Cell 4.")

print("\n--- 2 sample rows ---")
for a in audit[:2]:
    print("\n  {} [{} / {}] {} | grades {}".format(
        a["id"], a["language"], a["source"], a["difficulty"], a["grade"]))
    print("    Q  : {}".format(a["question"][:110]))
    print("    ref: {}".format(a["reference"][:110]))
    for k, v in a["answers"].items():
        print("    {:<8}: {}".format(k, v[:110] or "<empty>"))